# Glass IR — what the compiler actually emits

Gallowglass compiles to PLAN, but in between source and PLAN there's a typed intermediate representation called **Glass IR**. It carries the source `Loc`, the inferred type, the effect row, contract clauses, and a BLAKE3-256 pin hash that content-addresses the binding. It's the layer an editor / language server / MCP consumer of compiler output reads.

This notebook shows what Glass IR looks like for a few example declarations. The IR appears as static text in markdown cells — generated by a build script that runs the bootstrap pipeline and asks `bootstrap.glass_ir.render_fragment` for each fragment, so the pin hashes you see were computed against the current state of the compiler.

## A simple function

The Gallowglass source:

```gallowglass
let twice : Nat → Nat = λ n → n + n
```

compiles to this IR fragment:

```
-- Snapshot: pin#598059f6
-- Source: Demo.twice
-- Budget: 4096 tokens

-- @ <demo>:1:1
let Demo.twice [pin#598059f6] : Nat → Nat
  = λ n → n + n
```

Reading top-down:

* `pin#…` — first eight hex chars of the BLAKE3-256 of the compiled value. Two different bindings with structurally identical bodies share the same pin.
* `<demo>:1:1` — source location. Stays attached through the pipeline so error messages and tooling can map back.
* `Demo.twice [pin#…] : Nat → Nat` — fully-qualified name, pin reference, inferred type.
* `= λ n → n + n` — the body in canonical Gallowglass syntax. Round-trips: parsing this back produces the same PLAN value.

## A recursive function with pattern matching

The same shape, but with `fix` for self-recursion and a `match` body:

Source:

```gallowglass
let factorial : Nat → Nat
  = fix λ self n → match n {
      | 0 → 1
      | _ → n * self n
    }
```

IR:

```
-- Snapshot: pin#729da658
-- Source: Demo.factorial
-- Budget: 4096 tokens

-- @ <demo>:1:1
let Demo.factorial [pin#729da658] : Nat → Nat
  = fix λ self n → match n {
    | 0 → 1
    | _ → n * self n
  }
```

Two things to notice. First: the body in the IR is *not* a flat textual copy of the source — it's the result of running parse → scope → typecheck → render. So `self` is recovered from the `fix` introduction even though the compiled PLAN value uses a de Bruijn index for it.

Second: the pin hash captures the whole shape. Change the body in any way that affects the compiled PLAN value — add a guard, rename a binder, replace `*` with `+` — and the pin hash changes. Identical bodies in different modules get the same pin and dedupe in the emit layer.

## Pin hashes and content addressing

Pins are how cross-module deduplication works. When two modules contain bindings whose compiled values are bit-identical, the emit layer notices via pin equality and emits one Plan Asm `(#bind …)` for both. PR #81 in the project history was the dedup discipline that took the Compiler.gls emit from ~1.7 GB to ~775 KB.

## Where you actually see Glass IR

In day-to-day Gallowglass use you don't look at IR directly — the compiler does. But the layer is visible through:

* The MCP server (`python3 -m bootstrap.mcp_server`) — exposes `compile_snippet`, `infer_type`, and `render_fragment` over stdio for LLM consumers.
* The Jupyter kernel itself uses the typecheck output to drive the constructor-name renderer in cell output (the same mechanism that gives you `MkPair 3 7` instead of `((0 3) 7)`).

For the full IR grammar see `spec/01-glass-ir.md`.

## Verifying it lives — running a cell against the kernel

The IR you see above was generated against this same compile pipeline; the cell below evaluates the same function definition in the live kernel and calls it. The kernel's output uses the type-driven renderer, which itself reads the typecheck `con_info` and `type_env` that drives Glass IR rendering.

In [1]:
let twice : Nat → Nat = λ n → n + n

twice : Nat → Nat

In [2]:
twice 21

42

## What's next

The final lesson, `04-effects-and-handlers.ipynb`, covers Gallowglass's effect system — how effects appear in type signatures, the `eff` declaration, and the `handle` form. Running effectful computations from cells has a wrinkle the kernel doesn't fully expose yet, but the syntax itself is the part that matters for understanding the language.